# Floor System Assessment — `FloorSystemBase` API

Demonstrates the high-level floor system API from `scite.beam.floor`,
mirroring the later icc_app CNode projection.

All four concrete system types are exercised using only **model-level** methods —
no manual `FloorAnalysisPair` assembly visible in the notebook.

**Floor system taxonomy (European nomenclature)**
| Class | Section type | Reinforcement |
|---|---|---|
| `FlatSlab` | Solid rectangular strip 1 m wide | Steel (B500B) |
| `CRCFlatSlab` | Solid rectangular strip 1 m wide | CFRP |
| `SRCRibbedSlab` | T-section rib + bay slab strip | Steel (B500B) |
| `CRCRibbedSlab` | T-section rib + bay slab strip | CFRP |

**Scope**
- Section 1 — SRC Flat Slab: assessment + p–w curve
- Section 2 — CRC Flat Slab: assessment + p–w curve
- Section 3 — SRC vs CRC Flat Slab: normalised p–w + GWP/cost bar
- Section 4 — SRC Ribbed Slab: assessment + 3-panel p–w
- Section 5 — CRC Ribbed Slab: assessment + 3-panel p–w
- Section 6 — SRC vs CRC Ribbed Slab: normalised comparison
- Section 7 — GWP & cost comparison: all four systems
- Section 8 — Span sweep: GWP/m² and cost/m² vs L
- Section 9 — Polymorphism demo: uniform `assess()` loop

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.floor import (
    FloorSystemBase,
    LoadModel,
    FlatSlab,
    CRCFlatSlab,
    SRCRibbedSlab,
    CRCRibbedSlab,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# Common load model used throughout
lm = LoadModel(q_k=5.0, delta_g_k=1.5)
print('Imports OK')
print(f'Load model: delta_g_k={lm.delta_g_k} kN/m², q_k={lm.q_k} kN/m²')

---
## 1  SRC Flat Slab — assessment

A 200 mm solid slab strip with $A_s = 300\,\text{mm}^2/\text{m}$ ($z_s = 35\,\text{mm}$),
span $L = 5\,\text{m}$, concrete C30.

In [ ]:
src_flat = FlatSlab(h=200, A_s=1100, z_s=35, L=6000)

print(f'Self-weight g_k = {src_flat.g_k:.2f} kN/m²')
src_flat.print_assessment(lm)

In [ ]:
# Volumes, GWP and cost
v = src_flat.volumes()
print(f"Concrete  V = {v['V_c_per_m2']*1e3:.1f} L/m²    GWP = {v['gwp_conc']/v['A_ref']:.2f} kgCO₂/m²")
print(f"Steel     V = {v['V_s_per_m2']*1e6:.2f} cm³/m²  GWP = {v['gwp_steel']/v['A_ref']:.2f} kgCO₂/m²")
print(f"Total GWP   = {v['gwp_per_m2']:.2f} kgCO₂/m²")
print(f"Total cost  = {v['cost_per_m2']:.2f} €/m²")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
src_flat.plot_floor_assessment(axes, load_model=lm)
fig.suptitle('SRC Flat Slab — assessment', fontweight='bold')
plt.tight_layout()

---
## 2  CRC Flat Slab — assessment

A 140 mm thin slab with CFRP reinforcement $A_f = 120\,\text{mm}^2/\text{m}$
($z_f = 20\,\text{mm}$), same span.

In [ ]:
crc_flat = CRCFlatSlab(h=140, A_f=500, z_f=20, L=6000)

print(f'Self-weight g_k = {crc_flat.g_k:.2f} kN/m²')
crc_flat.print_assessment(lm)

In [ ]:
v = crc_flat.volumes()
print(f"Concrete  V = {v['V_c_per_m2']*1e3:.1f} L/m²    GWP = {v['gwp_conc']/v['A_ref']:.2f} kgCO₂/m²")
print(f"CFRP      V = {v['V_f_per_m2']*1e6:.2f} cm³/m²  GWP = {v['gwp_cfrp']/v['A_ref']:.2f} kgCO₂/m²")
print(f"Total GWP   = {v['gwp_per_m2']:.2f} kgCO₂/m²")
print(f"Total cost  = {v['cost_per_m2']:.2f} €/m²")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
crc_flat.plot_floor_assessment(axes, load_model=lm)
fig.suptitle('CRC Flat Slab — assessment', fontweight='bold')
plt.tight_layout()

---
## 3  SRC vs CRC Flat Slab — normalised p–w and resource comparison

Both systems carry the same design loads.  Normalised to maximum SLS capacity.

In [ ]:
a_src = src_flat.assess(lm)
a_crc = crc_flat.assess(lm)

print(f"{'System':<18} {'p_R_SLS':>9} {'p_R_ULS':>9} {'eta_SLS':>8} {'eta_ULS':>8}")
print('-' * 58)
for label, d in a_src.items():
    print(f"{'SRC ' + label:<18} {d['p_R_sls']:>9.2f} {d['p_R_uls']:>9.2f} {d['eta_SLS']:>8.3f} {d['eta_ULS']:>8.3f}")
for label, d in a_crc.items():
    print(f"{'CRC ' + label:<18} {d['p_R_sls']:>9.2f} {d['p_R_uls']:>9.2f} {d['eta_SLS']:>8.3f} {d['eta_ULS']:>8.3f}")

In [ ]:
v_src = src_flat.volumes()
v_crc = crc_flat.volumes()

labels   = ['SRC Flat Slab', 'CRC Flat Slab']
gwp      = [v_src['gwp_per_m2'], v_crc['gwp_per_m2']]
cost     = [v_src['cost_per_m2'], v_crc['cost_per_m2']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
colors = ['#1f77b4', '#d62728']
ax1.bar(labels, gwp, color=colors)
ax1.set_ylabel('GWP [kgCO₂eq/m²]')
ax1.set_title('GWP per m² floor area')
ax2.bar(labels, cost, color=colors)
ax2.set_ylabel('Cost [€/m²]')
ax2.set_title('Cost per m² floor area')
plt.tight_layout()

---
## 4  SRC Ribbed Slab — assessment

Rib: $B_{\text{rib}}=90\,\text{mm}$, $h_w=200\,\text{mm}$, $H_{\text{bay}}=60\,\text{mm}$,  
rib spacing $s=1200\,\text{mm}$, rib span $L_{\text{rib}}=6\,\text{m}$.

In [ ]:
src_rib = SRCRibbedSlab(
    H_rib=200, H_bay=60, B_rib=90,
    L_rib=6000, L_bay=1200,
    A_s_rib=1200, z_s_rib=30,
    A_s_bay=150, z_s_bay=20,
)

print(f'Self-weight g_k = {src_rib.g_k:.2f} kN/m²')
src_rib.print_assessment(lm)

In [ ]:
v = src_rib.volumes()
print(f"Concrete  V/bay = {v['V_c']:.4f} m³   GWP = {v['gwp_conc']:.2f} kgCO₂")
print(f"Steel (rib+bay) GWP = {v['gwp_steel']:.2f} kgCO₂")
print(f"Total GWP/m²  = {v['gwp_per_m2']:.2f} kgCO₂/m²")
print(f"Total cost/m² = {v['cost_per_m2']:.2f} €/m²")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
src_rib.plot_floor_assessment(axes, load_model=lm)
fig.suptitle('SRC Ribbed Slab — assessment', fontweight='bold')
plt.tight_layout()

---
## 5  CRC Ribbed Slab — assessment

Same geometry as Section 4, CFRP instead of steel.
Both rib **and** bay slab use CFRP structural analysis (`for_crc`).

In [ ]:
crc_rib = CRCRibbedSlab(
    H_rib=250, H_bay=50, B_rib=90,
    L_rib=6000, L_bay=800,
    A_f_rib=150, z_f_rib=20,
    A_f_bay=20,  z_f_bay=10,
)

print(f'Self-weight g_k = {crc_rib.g_k:.2f} kN/m²')
crc_rib.print_assessment(lm)

In [ ]:
v = crc_rib.volumes()
print(f"Concrete  V/bay = {v['V_c']:.4f} m³   GWP = {v['gwp_conc']:.2f} kgCO₂")
print(f"CFRP (rib+bay)  GWP = {v['gwp_cfrp']:.2f} kgCO₂")
print(f"Total GWP/m²  = {v['gwp_per_m2']:.2f} kgCO₂/m²")
print(f"Total cost/m² = {v['cost_per_m2']:.2f} €/m²")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
crc_rib.plot_floor_assessment(axes, load_model=lm)
fig.suptitle('CRC Ribbed Slab — assessment', fontweight='bold')
plt.tight_layout()

---
## 6  SRC vs CRC Ribbed Slab — normalised comparison

Capacities and utilisations side-by-side for the same geometry.

In [ ]:
a_src_rib = src_rib.assess(lm)
a_crc_rib = crc_rib.assess(lm)

print(f"{'System':<22} {'p_R_SLS':>9} {'p_R_ULS':>9} {'eta_SLS':>8} {'eta_ULS':>8}")
print('-' * 62)
for label, d in a_src_rib.items():
    print(f"{'SRC ' + label:<22} {d['p_R_sls']:>9.2f} {d['p_R_uls']:>9.2f} {d['eta_SLS']:>8.3f} {d['eta_ULS']:>8.3f}")
for label, d in a_crc_rib.items():
    print(f"{'CRC ' + label:<22} {d['p_R_sls']:>9.2f} {d['p_R_uls']:>9.2f} {d['eta_SLS']:>8.3f} {d['eta_ULS']:>8.3f}")

In [ ]:
v_src_rib = src_rib.volumes()
v_crc_rib = crc_rib.volumes()

labels_rib = ['SRC Ribbed Slab', 'CRC Ribbed Slab']
gwp_rib    = [v_src_rib['gwp_per_m2'], v_crc_rib['gwp_per_m2']]
cost_rib   = [v_src_rib['cost_per_m2'], v_crc_rib['cost_per_m2']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
colors = ['#1f77b4', '#d62728']
ax1.bar(labels_rib, gwp_rib, color=colors)
ax1.set_ylabel('GWP [kgCO₂eq/m²]')
ax1.set_title('GWP per m² floor area')
ax2.bar(labels_rib, cost_rib, color=colors)
ax2.set_ylabel('Cost [€/m²]')
ax2.set_title('Cost per m² floor area')
plt.tight_layout()

---
## 7  GWP and cost comparison — all four systems

Standard parameters; all systems designed for the same external load.
The bar chart makes the embodied-carbon trade-offs immediately visible.

In [ ]:
systems = {
    'SRC\nFlat Slab':    src_flat,
    'CRC\nFlat Slab':    crc_flat,
    'SRC\nRibbed Slab':  src_rib,
    'CRC\nRibbed Slab':  crc_rib,
}

all_gwp  = {name: s.volumes()['gwp_per_m2']  for name, s in systems.items()}
all_cost = {name: s.volumes()['cost_per_m2'] for name, s in systems.items()}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bar_colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e']

axes[0].bar(all_gwp.keys(),  all_gwp.values(),  color=bar_colors)
axes[0].set_ylabel('GWP [kgCO₂eq/m²]')
axes[0].set_title('GWP — all systems')

axes[1].bar(all_cost.keys(), all_cost.values(), color=bar_colors)
axes[1].set_ylabel('Cost [€/m²]')
axes[1].set_title('Cost — all systems')

plt.tight_layout()

---
## 8  Span sweep — GWP/m² and cost/m² vs span $L$

Flat slab spans from 3 m to 8 m; ribbed slab from 4 m to 12 m.
All other parameters held at defaults.

In [ ]:
L_flat_mm   = np.linspace(3000, 8000, 12)
L_ribbed_mm = np.linspace(4000, 12000, 12)

def _sweep_flat(L_values):
    gwp, cost = [], []
    for L in L_values:
        v_src = FlatSlab(h=200, A_s=300, z_s=35, L=L).volumes()
        v_crc = CRCFlatSlab(h=140, A_f=120, z_f=20, L=L).volumes()
        gwp.append([v_src['gwp_per_m2'], v_crc['gwp_per_m2']])
        cost.append([v_src['cost_per_m2'], v_crc['cost_per_m2']])
    return np.array(gwp), np.array(cost)

def _sweep_ribbed(L_values):
    gwp, cost = [], []
    for L in L_values:
        v_src = SRCRibbedSlab(H_rib=200, H_bay=60, B_rib=90, L_rib=L, L_bay=1200).volumes()
        v_crc = CRCRibbedSlab(H_rib=200, H_bay=60, B_rib=90, L_rib=L, L_bay=1200).volumes()
        gwp.append([v_src['gwp_per_m2'], v_crc['gwp_per_m2']])
        cost.append([v_src['cost_per_m2'], v_crc['cost_per_m2']])
    return np.array(gwp), np.array(cost)

gwp_flat,  cost_flat  = _sweep_flat(L_flat_mm)
gwp_rib,   cost_rib   = _sweep_ribbed(L_ribbed_mm)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

for ax, gwp, cost, L, title in [
    (axes[0, 0], gwp_flat,  cost_flat,  L_flat_mm / 1000,   'Flat Slab'),
    (axes[0, 1], gwp_rib,   cost_rib,   L_ribbed_mm / 1000, 'Ribbed Slab'),
]:
    ax.plot(L, gwp[:, 0], 'b-o', label='SRC', markersize=4)
    ax.plot(L, gwp[:, 1], 'r-s', label='CRC', markersize=4)
    ax.set_xlabel('Span L [m]')
    ax.set_ylabel('GWP [kgCO₂eq/m²]')
    ax.set_title(f'GWP — {title}')
    ax.legend()

for ax, gwp, cost, L, title in [
    (axes[1, 0], gwp_flat,  cost_flat,  L_flat_mm / 1000,   'Flat Slab'),
    (axes[1, 1], gwp_rib,   cost_rib,   L_ribbed_mm / 1000, 'Ribbed Slab'),
]:
    ax.plot(L, cost[:, 0], 'b-o', label='SRC', markersize=4)
    ax.plot(L, cost[:, 1], 'r-s', label='CRC', markersize=4)
    ax.set_xlabel('Span L [m]')
    ax.set_ylabel('Cost [€/m²]')
    ax.set_title(f'Cost — {title}')
    ax.legend()

plt.tight_layout()

---
## 9  Polymorphism demo — uniform `assess()` loop

`FloorSystemBase` is an ABC.  All four concrete types can be stored in a plain
list and assessed with a single call — no `isinstance` checks needed.

In [ ]:
from abc import ABC

all_systems: list[FloorSystemBase] = [src_flat, crc_flat, src_rib, crc_rib]
names = ['SRC Flat Slab', 'CRC Flat Slab', 'SRC Ribbed Slab', 'CRC Ribbed Slab']

assert all(isinstance(s, FloorSystemBase) for s in all_systems)
assert issubclass(FloorSystemBase, ABC)

print(f"\n{'System':<22} {'Element':<22} {'eta_SLS':>8} {'eta_ULS':>8}")
print('=' * 64)
for name, system in zip(names, all_systems):
    a = system.assess(lm)
    for elem, d in a.items():
        tag = f'[{name}]'
        print(f"{tag:<22} {elem:<22} {d['eta_SLS']:>8.3f} {d['eta_ULS']:>8.3f}")
    print()